# Google Colab Parallel 3 IDK Cascades

Clean real-time CPU/CUDA multiprocessing notebook for Google Colab. Model A runs on CPU; Model B and Model C run on the NVIDIA T4 CUDA GPU. Router cache data is regenerated in Colab.


## Colab Setup

This cell installs the package usually missing from Colab.


In [103]:
%pip install -q datasets


## Imports

This cell imports the libraries used by the notebook. The worker module is generated later so CUDA multiprocessing can import it with spawn.


In [104]:
import importlib
import json
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC


## Constants

Edit this cell to change models, devices, paths, worker counts, thresholds, sample counts, and output files. Model A runs on CPU; Model B and Model C run on CUDA.


In [105]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RUNS_DIR = PROJECT_ROOT / "runs"
ARTIFACTS_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet50"

MODEL_A_DEVICE = "cuda"
MODEL_B_DEVICE = "cuda"
MODEL_C_DEVICE = "cuda"

MODELS = (MODEL_A, MODEL_B, MODEL_C)
GPU_MODELS = (MODEL_B, MODEL_C)
MODEL_DEVICES = {
    MODEL_A: MODEL_A_DEVICE,
    MODEL_B: MODEL_B_DEVICE,
    MODEL_C: MODEL_C_DEVICE,
}

WORKER_LIMITS = {"cpu": 1, "cuda": 3}
MAX_SAMPLES = 10000
BATCH_SIZE = 1
CONFIDENCE_THRESHOLD = 0.9
MAX_IN_FLIGHT_PER_GPU_MODEL = 3

ROUTER_TRAIN_PREFIXES = ("matched", "top")
TEST_VARIANT = "threshold-0.7"
ROUTER_REQUIRES_CORRECT = False
SAVE_RESULTS = True
RESULTS_PATH = RUNS_DIR / "real_cpu_cuda_worker_results.json"
PREDICTIONS_PATH = RUNS_DIR / "real_cpu_cuda_worker_predictions.npz"


## CUDA Runtime Check

This cell verifies that Colab is using a GPU runtime.


In [106]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Colab, use Runtime > Change runtime type > T4 GPU.")

print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0))


CUDA available: True
CUDA device: Tesla T4


## ImageNetV2 Streaming Helper

This cell streams ImageNetV2 rows from Hugging Face without requiring the local `scripts/` folder.


In [107]:
VARIANT_URLS = {
    "matched-frequency": "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-threshold0.7.tar.gz",
    "top-images": "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-top-images.tar.gz",
}


def label_from_key(key):
    return int(key.split("/")[1])


def stream_imagenet_v2_rows(variant, max_samples=None):
    from datasets import load_dataset

    if variant not in VARIANT_URLS:
        choices = ", ".join(VARIANT_URLS)
        raise ValueError(f"Unknown variant '{variant}'. Choose one of: {choices}")

    stream = load_dataset(
        "webdataset",
        data_files={"train": VARIANT_URLS[variant]},
        split="train",
        streaming=True,
    )

    emitted = 0
    for row in stream:
        yield {
            "image": row["jpeg"].convert("RGB"),
            "label": label_from_key(row["__key__"]),
            "key": row["__key__"],
        }
        emitted += 1
        if max_samples is not None and emitted >= max_samples:
            break


## CUDA Worker Module

This cell writes the worker module used by multiprocessing. Spawned CUDA worker processes need an importable function instead of a notebook-local function.


In [108]:
WORKER_MODULE_PATH = PROJECT_ROOT / "colab_real_time_mp_workers.py"
WORKER_MODULE_PATH.write_text('import time\n\nimport torch\nfrom torchvision import models\n\n\nMODEL_SPECS = {\n    "resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),\n    "resnet34": (models.resnet34, models.ResNet34_Weights.DEFAULT),\n    "resnet50": (models.resnet50, models.ResNet50_Weights.DEFAULT),\n    "resnet152": (models.resnet152, models.ResNet152_Weights.DEFAULT),\n}\n\n\ndef model_worker(model_name, device_name, job_queue, result_queue):\n    try:\n        device = torch.device(device_name)\n        factory, weights = MODEL_SPECS[model_name]\n        model = factory(weights=weights).to(device).eval()\n\n        while True:\n            job = job_queue.get()\n            if job is None:\n                break\n\n            sample_index, images = job\n            start = time.perf_counter()\n            images = images.to(device)\n\n            with torch.inference_mode():\n                logits = model(images)\n                probabilities = torch.softmax(logits, dim=1)\n                if device.type == "cuda":\n                    torch.cuda.synchronize()\n                probabilities = probabilities[0].detach().cpu().numpy()\n\n            result_queue.put(\n                (\n                    sample_index,\n                    model_name,\n                    probabilities,\n                    int(probabilities.argmax()),\n                    float(probabilities.max()),\n                    (time.perf_counter() - start) * 1000.0,\n                )\n            )\n    except Exception as exc:\n        result_queue.put(("error", model_name, repr(exc)))\n', encoding="utf-8")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
import colab_real_time_mp_workers
importlib.reload(colab_real_time_mp_workers)
model_worker = colab_real_time_mp_workers.model_worker

print("Worker module:", WORKER_MODULE_PATH)


Worker module: /content/colab_real_time_mp_workers.py


## Image Transform

This cell defines the ImageNet preprocessing used for the real-time test images.


In [109]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


## Dataset Loader

This cell streams ImageNetV2 rows from Hugging Face, applies the ImageNet transform, and returns a DataLoader for the real-time test.


In [110]:
class StreamingImageNetV2Dataset(IterableDataset):
    def __init__(self, variant, max_samples):
        self.variant = variant
        self.max_samples = int(max_samples)

    def __iter__(self):
        for row in stream_imagenet_v2_rows(self.variant, self.max_samples):
            yield preprocess(row["image"]), int(row["label"])

    def __len__(self):
        return self.max_samples


def make_streaming_loader(variant, max_samples):
    dataset = StreamingImageNetV2Dataset(variant, max_samples)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
    )
    return dataset, loader


## Router Cache Validation

This cell finds the uploaded Colab artifacts folder and verifies that the six router cache files already exist.


In [111]:
REQUIRED_CACHE_FILES = [
    f"matched_{MODEL_A}.npz",
    f"matched_{MODEL_B}.npz",
    f"matched_{MODEL_C}.npz",
    f"top_{MODEL_A}.npz",
    f"top_{MODEL_B}.npz",
    f"top_{MODEL_C}.npz",
]

ARTIFACT_CANDIDATES = [
    Path("/content/artifacts"),
    Path("/content/Artifacts"),
    Path.cwd() / "artifacts",
    Path.cwd() / "Artifacts",
    PROJECT_ROOT / "artifacts",
    PROJECT_ROOT / "Artifacts",
]

SEARCH_ROOTS = [Path("/content"), Path.cwd(), PROJECT_ROOT]

def direct_missing(folder):
    return [name for name in REQUIRED_CACHE_FILES if not (folder / name).is_file()]


def unique_existing_paths(paths):
    seen = set()
    for path in paths:
        path = path.resolve()
        if path in seen or not path.exists():
            continue
        seen.add(path)
        yield path


def discover_cache_folders():
    folders = list(unique_existing_paths(ARTIFACT_CANDIDATES))
    for root in unique_existing_paths(SEARCH_ROOTS):
        for required_name in REQUIRED_CACHE_FILES:
            for found_file in root.rglob(required_name):
                parent = found_file.parent.resolve()
                if parent not in folders:
                    folders.append(parent)
    return folders


missing_by_candidate = {}
for candidate in discover_cache_folders():
    missing = direct_missing(candidate)
    if not missing:
        ARTIFACTS_DIR = candidate
        break
    missing_by_candidate[str(candidate)] = missing
else:
    found_npz = []
    for root in unique_existing_paths(SEARCH_ROOTS):
        found_npz.extend(str(path) for path in sorted(root.rglob("*.npz"))[:30])

    checked = "\n".join(
        f"{path}: missing {', '.join(missing)}"
        for path, missing in missing_by_candidate.items()
    )
    found = "\n".join(found_npz) if found_npz else "No .npz files found in this runtime."
    raise FileNotFoundError(
        "Could not find the six required cache files in this active Colab runtime.\n"
        "If VS Code shows the files under Colab CPU but this notebook runs on Colab GPU T4, "
        "upload the artifacts folder to Colab GPU T4 too. Colab runtimes do not share /content.\n\n"
        "Checked folders:\n"
        + (checked if checked else "No artifact-like folders found.")
        + "\n\nFound NPZ files in this runtime:\n"
        + found
    )

print("Using artifacts folder:", ARTIFACTS_DIR)
for name in REQUIRED_CACHE_FILES:
    size_mb = (ARTIFACTS_DIR / name).stat().st_size / (1024 * 1024)
    print(f"{name}: {size_mb:.1f} MB")


Using artifacts folder: /content/artifacts
matched_resnet18.npz: 36.5 MB
matched_resnet34.npz: 36.4 MB
matched_resnet50.npz: 34.2 MB
top_resnet18.npz: 36.5 MB
top_resnet34.npz: 36.4 MB
top_resnet50.npz: 34.1 MB


## Probability Features

This cell converts model probabilities into the three router features: confidence, entropy, and margin between the top two probabilities.


In [112]:
def probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)


def short_model_name(model_name):
    return model_name.replace("resnet", "RN").upper()


## Router Training Data

This cell builds router training data from the regenerated artifact NPZ files. It uses Model A probabilities as features and labels uncertain samples for Model B or Model C.


In [113]:
def load_cache(prefix, model_name):
    path = ARTIFACTS_DIR / f"{prefix}_{model_name}.npz"
    with np.load(path) as data:
        return {
            "probabilities": data["probabilities"],
            "predictions": data["predictions"],
            "labels": data["labels"],
        }


def cache_confidence(cache):
    return np.asarray(cache["probabilities"]).max(axis=1)


def build_router_training_data():
    feature_parts = []
    label_parts = []

    for prefix in ROUTER_TRAIN_PREFIXES:
        cache_a = load_cache(prefix, MODEL_A)
        cache_b = load_cache(prefix, MODEL_B)
        cache_c = load_cache(prefix, MODEL_C)

        features_a = probability_features(cache_a["probabilities"])
        model_a_uncertain = features_a[:, 0] < CONFIDENCE_THRESHOLD
        model_b_ok = cache_confidence(cache_b) >= CONFIDENCE_THRESHOLD
        model_c_ok = cache_confidence(cache_c) >= CONFIDENCE_THRESHOLD

        if ROUTER_REQUIRES_CORRECT:
            model_b_ok = model_b_ok & (cache_b["predictions"] == cache_a["labels"])
            model_c_ok = model_c_ok & (cache_c["predictions"] == cache_a["labels"])

        keep = model_a_uncertain & (model_b_ok | model_c_ok)
        route_labels = np.where(model_b_ok[keep], 0, 1).astype(np.int64)

        feature_parts.append(features_a[keep])
        label_parts.append(route_labels)

    router_features = np.concatenate(feature_parts)
    router_labels = np.concatenate(label_parts)
    if len(router_labels) == 0:
        raise ValueError("No router training rows found")
    return router_features, router_labels


router_train_features, router_train_labels = build_router_training_data()
print("Router training samples:", len(router_train_labels))
print(f"{MODEL_B} labels:", int(np.count_nonzero(router_train_labels == 0)))
print(f"{MODEL_C} labels:", int(np.count_nonzero(router_train_labels == 1)))


Router training samples: 2328
resnet34 labels: 2311
resnet50 labels: 17


## Random Forest Router

Run this cell to use a Random Forest router. To switch routers during Run All, comment out this cell and uncomment one of the next router cells.


In [114]:
# router_name = "rf"
# router = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=6,
#     min_samples_leaf=20,
#     class_weight="balanced",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## Extra Trees Router

Uncomment and run this cell to use an Extra Trees router. Comment out the other router cells first.


In [115]:
# router_name = "extratree"
# router = ExtraTreesClassifier(
#     n_estimators=100,
#     max_depth=6,
#     min_samples_leaf=20,
#     class_weight="balanced",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## XGBoost Router

Uncomment and run this cell to use an XGBoost router. Install xgboost before using it.


In [ ]:
# Install xgboost before uncommenting this cell.
# from xgboost import XGBClassifier

# router_name = "xgboost"
# router = XGBClassifier(
#     n_estimators=100,
#     max_depth=4,
#     learning_rate=0.1,
#     eval_metric="logloss",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


Router: xgboost


## SVM Router

Uncomment and run this cell to use an SVM router. Comment out the other router cells first.


In [123]:
router_name = "svm"
router = SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced")
router.fit(router_train_features, router_train_labels)
print("Router:", router_name)


Router: svm


## Test Dataset

This cell creates the Hugging Face streaming test loader and prints the selected worker devices.


In [124]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required because Model B and Model C are configured for CUDA")

real_test_dataset, real_test_loader = make_streaming_loader(TEST_VARIANT, MAX_SAMPLES)
real_sample_count = len(real_test_dataset)

print("Streaming samples:", real_sample_count)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Worker limits:", WORKER_LIMITS)
print("Worker model devices:", MODEL_DEVICES)


Streaming samples: 10000
CUDA device: Tesla T4
Worker limits: {'cpu': 1, 'cuda': 3}
Worker model devices: {'resnet18': 'cuda', 'resnet34': 'cuda', 'resnet50': 'cuda'}


## Doubly Linked List

This cell defines the heavy-model waiting queue. Model B jobs stay at the head side; Model C jobs stay at the tail side; Model C can steal a Model B job when no Model C job is waiting.


In [125]:
class JobNode:
    def __init__(self, sample_index, images, assigned_model):
        self.sample_index = sample_index
        self.images = images
        self.assigned_model = assigned_model
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self, model_b, model_c):
        self.model_b = model_b
        self.model_c = model_c
        self.head = None
        self.tail = None
        self.last_model_b = None
        self.size = 0

    def _insert_empty(self, node):
        self.head = node
        self.tail = node
        self.size = 1
        if node.assigned_model == self.model_b:
            self.last_model_b = node

    def _insert_before(self, anchor, node):
        node.prev = anchor.prev
        node.next = anchor
        if anchor.prev is None:
            self.head = node
        else:
            anchor.prev.next = node
        anchor.prev = node
        self.size += 1

    def _insert_after(self, anchor, node):
        node.prev = anchor
        node.next = anchor.next
        if anchor.next is None:
            self.tail = node
        else:
            anchor.next.prev = node
        anchor.next = node
        self.size += 1

    def _remove(self, node):
        if node.prev is None:
            self.head = node.next
        else:
            node.prev.next = node.next

        if node.next is None:
            self.tail = node.prev
        else:
            node.next.prev = node.prev

        if node is self.last_model_b:
            self.last_model_b = node.prev if node.prev and node.prev.assigned_model == self.model_b else None

        node.prev = None
        node.next = None
        self.size -= 1

        if self.size == 0:
            self.head = None
            self.tail = None
            self.last_model_b = None

        return node

    def insert_middle(self, sample_index, images, assigned_model):
        node = JobNode(sample_index, images, assigned_model)
        if self.size == 0:
            self._insert_empty(node)
            return node

        if assigned_model == self.model_b:
            if self.last_model_b is None:
                self._insert_before(self.head, node)
            else:
                self._insert_after(self.last_model_b, node)
            self.last_model_b = node
        elif assigned_model == self.model_c:
            if self.last_model_b is None:
                self._insert_before(self.head, node)
            else:
                self._insert_after(self.last_model_b, node)
        else:
            raise ValueError(f"Unknown heavy model: {assigned_model}")

        return node

    def pop_for_model_b(self):
        if self.head is None or self.head.assigned_model != self.model_b:
            return None
        return self._remove(self.head)

    def pop_for_model_c_or_steal_model_b(self):
        if self.tail is not None and self.tail.assigned_model == self.model_c:
            return self._remove(self.tail), False

        node = self.pop_for_model_b()
        if node is None:
            return None, False
        return node, True


## Router Function

This cell converts Model A probabilities into router features and returns Model B or Model C.


In [126]:
def route_from_model_a(probabilities):
    route_label = int(router.predict(probability_features(probabilities[None, :]))[0])
    return MODEL_B if route_label == 0 else MODEL_C


## Real-Time Run

This cell starts one CPU worker for Model A and two CUDA workers for Model B and Model C, routes uncertain Model A samples, and records all metrics.


In [127]:
def run_real_time_test():
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    job_queues = {model_name: ctx.SimpleQueue() for model_name in MODELS}
    result_queue = ctx.SimpleQueue()
    processes = [
        ctx.Process(
            target=model_worker,
            args=(model_name, MODEL_DEVICES[model_name], job_queues[model_name], result_queue),
        )
        for model_name in MODELS
    ]

    for process in processes:
        process.start()

    try:
        total_samples = len(real_test_dataset)
        labels = np.full(total_samples, -1, dtype=np.int64)
        final_predictions = np.full(total_samples, -1, dtype=np.int64)
        chosen_models = np.full(total_samples, "", dtype="<U32")
        latencies_ms = np.full(total_samples, np.nan, dtype=np.float64)

        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        heavy_route_count_by_model = Counter()
        stolen_job_count_by_model = Counter()
        in_flight_by_model = Counter()

        sample_states = {}
        doubly_linked_list = DoublyLinkedList(MODEL_B, MODEL_C)
        loader_iterator = iter(real_test_loader)
        next_sample_index = 0
        completed_sample_count = 0
        heavy_queue_max_size = 0
        router_call_count = 0
        router_time_ms_total = 0.0
        run_start = time.perf_counter()

        def heavy_backlog_size():
            return doubly_linked_list.size + sum(in_flight_by_model[model_name] for model_name in GPU_MODELS)

        def finish_sample(sample_index, model_name, prediction):
            final_predictions[sample_index] = prediction
            chosen_models[sample_index] = model_name
            latencies_ms[sample_index] = (time.perf_counter() - sample_states[sample_index]["start_time"]) * 1000.0
            sample_states.pop(sample_index)

        def dispatch_gpu_jobs(max_size):
            for model_name in GPU_MODELS:
                while in_flight_by_model[model_name] < MAX_IN_FLIGHT_PER_GPU_MODEL:
                    if model_name == MODEL_B:
                        node = doubly_linked_list.pop_for_model_b()
                        stole_model_b_job = False
                    else:
                        node, stole_model_b_job = doubly_linked_list.pop_for_model_c_or_steal_model_b()

                    if node is None:
                        break

                    job_queues[model_name].put((node.sample_index, node.images))
                    in_flight_by_model[model_name] += 1
                    execution_count_by_model[model_name] += 1

                    if stole_model_b_job:
                        stolen_job_count_by_model[MODEL_C] += 1

                    max_size = max(max_size, heavy_backlog_size())
            return max_size

        while completed_sample_count < total_samples:
            while in_flight_by_model[MODEL_A] < WORKER_LIMITS["cpu"] and next_sample_index < total_samples:
                images, batch_labels = next(loader_iterator)
                sample_index = next_sample_index
                next_sample_index += 1
                labels[sample_index] = int(batch_labels.item())
                sample_states[sample_index] = {"images": images, "start_time": time.perf_counter()}
                job_queues[MODEL_A].put((sample_index, images))
                in_flight_by_model[MODEL_A] += 1
                execution_count_by_model[MODEL_A] += 1

            message = result_queue.get()
            if message[0] == "error":
                _, model_name, error = message
                raise RuntimeError(f"{model_name} worker failed: {error}")

            sample_index, model_name, probabilities, prediction, confidence, elapsed_ms = message
            in_flight_by_model[model_name] -= 1
            execution_time_ms_by_model[model_name] += elapsed_ms

            if model_name == MODEL_A and confidence < CONFIDENCE_THRESHOLD:
                router_start = time.perf_counter()
                assigned_model = route_from_model_a(probabilities)
                router_time_ms_total += (time.perf_counter() - router_start) * 1000.0
                router_call_count += 1
                heavy_route_count_by_model[assigned_model] += 1
                doubly_linked_list.insert_middle(sample_index, sample_states[sample_index]["images"], assigned_model)
                sample_states[sample_index]["images"] = None
                heavy_queue_max_size = max(heavy_queue_max_size, heavy_backlog_size())
            else:
                finish_sample(sample_index, model_name, prediction)
                completed_sample_count += 1

            heavy_queue_max_size = dispatch_gpu_jobs(heavy_queue_max_size)

        total_wall_time_seconds = time.perf_counter() - run_start
        correct_predictions = int(np.count_nonzero(final_predictions == labels))
        total_execution_time_ms_by_model = {model_name: float(execution_time_ms_by_model[model_name]) for model_name in MODELS}
        mean_execution_time_ms_by_model = {
            model_name: float(execution_time_ms_by_model[model_name] / execution_count_by_model[model_name])
            if execution_count_by_model[model_name]
            else 0.0
            for model_name in MODELS
        }

        results = {
            "total_samples": int(total_samples),
            "accuracy": float(correct_predictions / total_samples),
            "correct_predictions": int(correct_predictions),
            "total_wall_time_seconds": float(total_wall_time_seconds),
            "throughput_fps": float(total_samples / total_wall_time_seconds),
            "mean_latency_ms": float(latencies_ms.mean()),
            "worker_limits": dict(WORKER_LIMITS),
            "device_by_model": {model_name: MODEL_DEVICES[model_name] for model_name in MODELS},
            "router_name": router_name,
            "router_call_count": int(router_call_count),
            "total_router_time_ms": float(router_time_ms_total),
            "mean_router_time_ms": float(router_time_ms_total / router_call_count) if router_call_count else 0.0,
            "stolen_job_count_by_model": {model_name: int(stolen_job_count_by_model[model_name]) for model_name in MODELS},
            "final_prediction_count_by_model": {model_name: int(np.count_nonzero(chosen_models == model_name)) for model_name in MODELS},
            "execution_count_by_model": {model_name: int(execution_count_by_model[model_name]) for model_name in MODELS},
            "mean_execution_time_ms_by_model": mean_execution_time_ms_by_model,
            "idle_time_ms_by_model": {
                model_name: max(0.0, total_wall_time_seconds * 1000.0 - total_execution_time_ms_by_model[model_name])
                for model_name in MODELS
            },
            "heavy_route_count_by_model": {model_name: int(heavy_route_count_by_model[model_name]) for model_name in GPU_MODELS},
            "heavy_queue_max_size": int(heavy_queue_max_size),
        }
        return results, final_predictions, chosen_models

    finally:
        for queue in job_queues.values():
            queue.put(None)
        for process in processes:
            process.join()


real_system_results, real_final_predictions, real_chosen_models = run_real_time_test()


## Print and Save Metrics

This cell prints the requested real-time metrics and saves the JSON results plus NPZ predictions when SAVE_RESULTS is True.


In [128]:
print("Real-Time CPU/CUDA Multiprocessing Test")
print("Confidence Threshold Constant: ", CONFIDENCE_THRESHOLD)
print("Max in flight: ", MAX_IN_FLIGHT_PER_GPU_MODEL)
print("Worker limits:", real_system_results["worker_limits"])
print("Device by model:", real_system_results["device_by_model"])
print("Total samples:", real_system_results["total_samples"])
print("Accuracy:", round(real_system_results["accuracy"], 4))
print("Correct predictions:", real_system_results["correct_predictions"])
print("Total wall time (seconds):", round(real_system_results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(real_system_results["throughput_fps"], 3))
print("Mean latency (ms):", round(real_system_results["mean_latency_ms"], 3))
print()
print(f"Classifier {real_system_results['router_name']} router:")
print(f"  Router calls: {real_system_results['router_call_count']}")
print(f"  Total router time (ms): {real_system_results['total_router_time_ms']:.3f}")
print(f"  Mean router time (ms): {real_system_results['mean_router_time_ms']:.6f}")
print()
print(
    f"# of jobs stolen by {short_model_name(MODEL_C)} originally assigned to {short_model_name(MODEL_B)}:",
    real_system_results["stolen_job_count_by_model"][MODEL_C],
)
print()
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['execution_count_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['mean_execution_time_ms_by_model'][model_name]:.3f}")
print("Idle time by model (seconds):")
for model_name in MODELS:
    idle_seconds = real_system_results["idle_time_ms_by_model"][model_name] / 1000.0
    print(f"  {model_name}: {idle_seconds:.3f}")
print("Heavy route count by CUDA model:")
for model_name in GPU_MODELS:
    print(f"  {model_name}: {real_system_results['heavy_route_count_by_model'][model_name]}")
print("Heavy queue max size:", real_system_results["heavy_queue_max_size"])

if SAVE_RESULTS:
    RESULTS_PATH.write_text(json.dumps(real_system_results, indent=2) + "\n", encoding="utf-8")
    np.savez_compressed(
        PREDICTIONS_PATH,
        predictions=real_final_predictions,
        chosen_models=real_chosen_models,
    )
    print("Saved:", RESULTS_PATH.name)
    print("Saved:", PREDICTIONS_PATH.name)
else:
    print("SAVE_RESULTS is False; no files were written.")


Real-Time CPU/CUDA Multiprocessing Test
Confidence Threshold Constant:  0.9
Max in flight:  3
Worker limits: {'cpu': 1, 'cuda': 3}
Device by model: {'resnet18': 'cuda', 'resnet34': 'cuda', 'resnet50': 'cuda'}
Total samples: 10000
Accuracy: 0.7246
Correct predictions: 7246
Total wall time (seconds): 177.71
Throughput (FPS): 56.272
Mean latency (ms): 16.89

Classifier svm router:
  Router calls: 6278
  Total router time (ms): 4218.913
  Mean router time (ms): 0.672015

# of jobs stolen by RN50 originally assigned to RN34: 1

Final prediction count by model:
  resnet18: 3722
  resnet34: 4593
  resnet50: 1685
Execution count by model:
  resnet18: 10000
  resnet34: 4593
  resnet50: 1685
Mean execution time by model (ms):
  resnet18: 4.082
  resnet34: 8.651
  resnet50: 13.687
Idle time by model (seconds):
  resnet18: 136.886
  resnet34: 137.976
  resnet50: 154.647
Heavy route count by CUDA model:
  resnet34: 4594
  resnet50: 1684
Heavy queue max size: 7
Saved: real_cpu_cuda_worker_results.js